# Normalização e padronização de dados

Antes de treinar determinados modelos de aprendizado de máquina, é importante observar a escala das variáveis. Uma característica pode assumir valores pequenos, enquanto outra pode ter valores muito maiores. Quando o algoritmo utiliza distâncias ou é sensível à magnitude das variáveis, essa diferença pode influenciar bastante o treinamento.

Neste notebook, vamos comparar duas técnicas muito utilizadas:

- **Min-Max Scaling (`MinMaxScaler`)**: transforma os valores para um intervalo definido, por padrão de 0 a 1.
- **Padronização (`StandardScaler`)**: transforma os valores para que tenham média próxima de 0 e desvio padrão próximo de 1.

Usaremos o conjunto de dados **Wine**, disponível no scikit-learn, e compararemos as variáveis `alcohol` e `proline`.

## Por que ajustar a escala das variáveis?

Imagine um conjunto de dados com duas características: uma varia aproximadamente entre 0 e 1, enquanto outra varia entre centenas e milhares. Em modelos que calculam distâncias, a variável com valores maiores pode dominar o cálculo e influenciar desproporcionalmente o resultado.

Ajustar as escalas pode ajudar a:
- evitar que a magnitude numérica de uma variável domine outras;
- tornar as distâncias mais comparáveis entre características;
- melhorar o comportamento de algoritmos sensíveis à escala, como K-Means, K-vizinhos mais próximos (KNN) e métodos baseados em gradiente.

Nem todo modelo exige escalonamento. Árvores de decisão e modelos baseados em árvores, por exemplo, geralmente são menos sensíveis à escala das variáveis.

> **Atenção:** escalonar os dados não significa que eles passam a ter mais informação. A transformação altera a escala numérica, não o conteúdo essencial da variável.

## Carregando os dados

O conjunto Wine contém medidas químicas de vinhos, além de um rótulo de classe. Para este exemplo, vamos manter a classe no DataFrame para contextualizar a base, mas aplicaremos os escaladores apenas às características escolhidas.

In [ ]:
from sklearn.datasets import load_wine
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler

dataset = load_wine()

x = dataset.data
x_cols = dataset.feature_names
y = dataset.target

df = pd.DataFrame(data=x, columns=x_cols)
df["class"] = y

df.head()

## Comparando as escalas originais

Vamos observar as estatísticas descritivas de `alcohol` e `proline`. Elas têm unidades e amplitudes diferentes, algo que também fica visível no gráfico.

In [ ]:
df[["alcohol", "proline"]].describe().round(2)

In [ ]:
samples = np.linspace(1, len(df), len(df))

plt.figure(figsize=(10, 6))
plt.scatter(x=samples, y=df["alcohol"], label="álcool", color="purple")
plt.scatter(x=samples, y=df["proline"], label="proline", color="green")
plt.xlabel("Amostra")
plt.ylabel("Valor original")
plt.title("Variáveis antes do escalonamento")
plt.legend()
plt.show()

No gráfico, a escala de `proline` é muito maior do que a de `alcohol`. Como ambas compartilham o mesmo eixo vertical, os valores de `alcohol` ficam comprimidos visualmente perto da parte inferior.

Isso ilustra uma diferença de magnitude, mas não significa que `proline` seja necessariamente mais importante para o problema. A relevância depende do objetivo e do modelo utilizado.

## Min-Max Scaling

O `MinMaxScaler` transforma cada característica para um intervalo definido. Por padrão, o menor valor observado na característica passa a ser 0 e o maior passa a ser 1. Os demais valores são posicionados proporcionalmente entre esses extremos.

A transformação padrão pode ser representada por:

$
x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}
$

Esse método preserva a ordem dos valores e mantém a distribuição relativa entre mínimo e máximo, mas é sensível a valores extremos (*outliers*): um valor muito alto ou muito baixo pode comprimir a maior parte dos dados no intervalo.

In [ ]:
scaler = MinMaxScaler()

alcool_scaled = scaler.fit_transform(df[["alcohol"]])
proline_scaled = scaler.fit_transform(df[["proline"]])

df_minmax = df.copy()
df_minmax["alcohol_minmax"] = alcool_scaled
df_minmax["proline_minmax"] = proline_scaled

df_minmax[["alcohol_minmax", "proline_minmax"]].describe().round(3)

Observe que usamos `fit_transform` em cada característica separadamente. O método `fit` aprende os valores necessários para a transformação (como mínimo e máximo); o método `transform` aplica a transformação aos dados.

Para demonstrar a comparação visual, vamos desenhar as duas variáveis já transformadas no mesmo gráfico.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(x=samples, y=alcool_scaled.ravel(), label="álcool", color="purple")
plt.scatter(x=samples, y=proline_scaled.ravel(), label="proline", color="green")
plt.xlabel("Amostra")
plt.ylabel("Valor escalonado")
plt.title("Variáveis após Min-Max Scaling")
plt.legend()
plt.show()

Agora as duas características ficam entre 0 e 1, facilitando a comparação visual da variação relativa de cada uma. Isso não significa que os valores transformados das duas variáveis passem a representar a mesma coisa: cada uma continua medindo uma característica química diferente.

## Standard Scaling (padronização)

O `StandardScaler` centraliza cada característica subtraindo sua média e, em seguida, divide pelo desvio padrão. O resultado tem média próxima de 0 e desvio padrão próximo de 1 nos dados usados para ajustar o escalador.

A transformação pode ser representada por:

$
z = \frac{x - \mu}{\sigma}
$

Em que:
- $x$ é o valor original;
- $\mu$ é a média da característica;
- $\sigma$ é o desvio padrão da característica.

Após a padronização, os valores podem ser negativos ou positivos. Um valor 0 indica que a observação está na média da característica; um valor positivo está acima da média, e um valor negativo está abaixo da média.

O `StandardScaler` também pode ser afetado por valores extremos, pois a média e o desvio padrão são sensíveis a eles.

In [ ]:
scaler = StandardScaler()

alcool_scaled_std = scaler.fit_transform(df[["alcohol"]])
proline_scaled_std = scaler.fit_transform(df[["proline"]])

df_standard = df.copy()
df_standard["alcohol_standard"] = alcool_scaled_std
df_standard["proline_standard"] = proline_scaled_std

df_standard[["alcohol_standard", "proline_standard"]].describe().round(3)

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(x=samples, y=alcool_scaled_std.ravel(), label="álcool", color="purple")
plt.scatter(x=samples, y=proline_scaled_std.ravel(), label="proline", color="green")
plt.axhline(0, linestyle="--", linewidth=1)
plt.xlabel("Amostra")
plt.ylabel("Valor padronizado (z-score)")
plt.title("Variáveis após Standard Scaling")
plt.legend()
plt.show()

O gráfico agora mostra os valores em relação à média de cada característica. A posição acima ou abaixo de zero indica se cada observação está acima ou abaixo da média correspondente.

Diferentemente do Min-Max Scaling, a padronização não força os valores a ficarem entre 0 e 1: é normal observar valores negativos e positivos e, ocasionalmente, valores com magnitude superior a 1.

## Comparando as duas técnicas

| Característica | Min-Max Scaling | Standard Scaling |
|---|---|---|
| Transformação | Reescala para um intervalo | Centraliza pela média e divide pelo desvio padrão |
| Escala padrão | Entre 0 e 1 | Média próxima de 0 e desvio padrão próximo de 1 |
| Valores negativos | Não aparecem com o intervalo padrão, salvo situações fora do intervalo observado | Podem aparecer normalmente |
| Sensibilidade a outliers | Sensível | Também sensível |
| Uso típico | Quando um intervalo limitado é desejável | Quando se deseja centralizar e padronizar a dispersão |

Não existe uma técnica universalmente melhor. A escolha depende dos dados, do algoritmo e do objetivo da aplicação. Em projetos reais, é comum testar alternativas e avaliar o desempenho do modelo.

## Conclusão

- Variáveis em escalas muito diferentes podem influenciar de forma desigual modelos sensíveis à magnitude ou à distância.
- `MinMaxScaler` coloca os dados, por padrão, no intervalo de 0 a 1.
- `StandardScaler` centraliza as características e as transforma para uma escala baseada no desvio padrão.
- As duas técnicas podem ser sensíveis a valores extremos.
- Em avaliações com treino e teste, ajuste o escalador somente no treino e aplique `transform` aos demais conjuntos.
- O escalonamento não é obrigatório para todos os algoritmos; a escolha deve considerar o modelo e os dados.

Neste notebook, não aplicamos escalonamento ao conjunto inteiro para treinar um modelo. O objetivo foi compreender as transformações e aprender o fluxo correto de preparação dos dados.